In [1]:
"""
=============================================================
Phase 3 — Step 1: Download ERA5-Land + คำนวณ ET₀ รายสัปดาห์
Mae Na Rua Sub-District, Phayao (19.05°N, 99.80°E)
Period: 2020–2024
=============================================================
PREREQUISITES:
  pip install cdsapi xarray netCDF4 scipy pandas numpy

CDS API KEY SETUP (ทำครั้งเดียว):
  1. สร้างไฟล์ C:/Users/<username>/.cdsapirc  (Windows)
     หรือ ~/.cdsapirc  (Mac/Linux)
  2. ใส่เนื้อหา:
       url: https://cds.climate.copernicus.eu/api
       key: <your-api-key-here>
=============================================================
"""

# ── PART A: ตรวจสอบ API key ก่อน ─────────────────────────────
import cdsapi
import os

print("=== ตรวจสอบ CDS API ===")
try:
    c = cdsapi.Client()
    print("✅ CDS API พร้อมใช้งาน")
except Exception as e:
    print(f"❌ CDS API Error: {e}")
    print("""
แก้ไข:
1. สร้างไฟล์ C:/Users/<ชื่อผู้ใช้>/.cdsapirc
2. ใส่เนื้อหา (แทน YOUR_KEY ด้วย key จริง):
   url: https://cds.climate.copernicus.eu/api
   key: YOUR_KEY
3. Restart kernel แล้วรันใหม่
    """)
    raise




=== ตรวจสอบ CDS API ===
✅ CDS API พร้อมใช้งาน


In [3]:
import cdsapi
import zipfile
import os
import shutil

c = cdsapi.Client()

variables = [
    '2m_temperature', '2m_dewpoint_temperature',
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'surface_net_solar_radiation', 'surface_net_thermal_radiation',
    'total_precipitation',
]

DST_DIR = r'c:\Users\mpdox\Desktop\ETo'
os.makedirs(DST_DIR, exist_ok=True)

for year in [2020, 2021, 2022, 2023, 2024]:
    final_nc  = os.path.join(DST_DIR, f'era5_{year}.nc')
    temp_zip  = os.path.join(DST_DIR, f'era5_{year}_raw.zip')

    if os.path.exists(final_nc):
        size = os.path.getsize(final_nc)
        if size > 1_000_000:   # > 1 MB = สมบูรณ์
            print(f"⚡ {year}: มีอยู่แล้ว ({size/1e6:.1f} MB) — ข้าม")
            continue
        else:
            print(f"⚠️  {year}: ไฟล์เล็กเกินไป ({size:,} bytes) — re-download")
            os.remove(final_nc)

    print(f"\n📥 Downloading {year}...")
    c.retrieve(
        'reanalysis-era5-land',
        {
            'variable':     variables,
            'product_type': 'reanalysis',
            'year':         str(year),
            'month':        [str(m).zfill(2) for m in range(1, 13)],
            'day':          [str(d).zfill(2) for d in range(1, 32)],
            'time':         '12:00',
            'area':         [19.3, 99.5, 18.9, 100.1],
            'format':       'netcdf',
        },
        temp_zip   # ← บันทึกเป็น temp ก่อน
    )

    # ตรวจสอบว่าเป็น ZIP หรือ NC
    with open(temp_zip, 'rb') as f:
        header = f.read(4)

    if header[:2] == b'PK':   # ZIP
        print(f"  Unzip {year}...")
        with zipfile.ZipFile(temp_zip, 'r') as z:
            # extract data_0.nc แล้ว rename
            z.extract('data_0.nc', DST_DIR)
            os.rename(
                os.path.join(DST_DIR, 'data_0.nc'),
                final_nc
            )
        os.remove(temp_zip)
    else:
        # เป็น NC โดยตรง
        os.rename(temp_zip, final_nc)

    size = os.path.getsize(final_nc)
    print(f"  ✅ era5_{year}.nc saved ({size/1e6:.1f} MB)")



⚠️  2020: ไฟล์เล็กเกินไป (318,978 bytes) — re-download


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\mpdox\\Desktop\\ETo\\era5_2020.nc'

In [4]:
# ── ตรวจสอบทุกไฟล์ ────────────────────────────────────────────
DST_DIR = r'c:\Users\mpdox\Desktop\ETo'
os.makedirs(DST_DIR, exist_ok=True)

print("\n=== ไฟล์ที่ได้ ===")
for year in [2020, 2021, 2022, 2023, 2024]:
    path = os.path.join(DST_DIR, f'era5_{year}.nc')
    if os.path.exists(path):
        size = os.path.getsize(path)
        status = "✅" if size > 1_000_000 else "❌ เล็กเกินไป"
        print(f"  era5_{year}.nc : {size/1e6:.1f} MB  {status}")
    else:
        print(f"  era5_{year}.nc : ❌ ไม่พบ")

# ── ทดสอบเปิดไฟล์ ─────────────────────────────────────────────
print("\n=== ทดสอบเปิด era5_2020.nc ===")
import xarray as xr
ds = xr.open_dataset(
    os.path.join(DST_DIR, 'era5_2020.nc'),
    engine='netcdf4'
)
print("Variables :", list(ds.data_vars))
print("Time      :", str(ds.time.values[0])[:10],
      "→", str(ds.time.values[-1])[:10])
print("Lat       :", ds.latitude.values)
print("Lon       :", ds.longitude.values)
print("t2m sample:", float(ds['t2m'].isel(time=0).values.flat[0]), "K")
ds.close()
print("✅ ไฟล์ใช้งานได้")


=== ไฟล์ที่ได้ ===
  era5_2020.nc : 0.3 MB  ❌ เล็กเกินไป
  era5_2021.nc : 0.3 MB  ❌ เล็กเกินไป
  era5_2022.nc : 0.3 MB  ❌ เล็กเกินไป
  era5_2023.nc : 0.3 MB  ❌ เล็กเกินไป
  era5_2024.nc : 0.3 MB  ❌ เล็กเกินไป

=== ทดสอบเปิด era5_2020.nc ===
Variables : ['t2m', 'd2m', 'u10', 'v10', 'ssr', 'str', 'tp']


AttributeError: 'Dataset' object has no attribute 'time'

In [5]:
import xarray as xr, os

DST_DIR = r'c:\Users\mpdox\Desktop\ETo'
os.makedirs(DST_DIR, exist_ok=True)

ds = xr.open_dataset(os.path.join(DST_DIR, 'era5_2020.nc'), engine='netcdf4')

print("Variables :", list(ds.data_vars))
print("Dimensions:", dict(ds.dims))
print("Coords    :", list(ds.coords))
print(ds)  # ดู full structure

Variables : ['t2m', 'd2m', 'u10', 'v10', 'ssr', 'str', 'tp']
Dimensions: {'valid_time': 366, 'latitude': 5, 'longitude': 7}
Coords    : ['number', 'valid_time', 'latitude', 'longitude', 'expver']
<xarray.Dataset> Size: 368kB
Dimensions:     (valid_time: 366, latitude: 5, longitude: 7)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2020-01-01T12:00:00 ... 2020-...
    expver      (valid_time) <U4 6kB ...
  * latitude    (latitude) float64 40B 19.3 19.2 19.1 19.0 18.9
  * longitude   (longitude) float64 56B 99.5 99.6 99.7 99.8 99.9 100.0 100.1
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 51kB ...
    d2m         (valid_time, latitude, longitude) float32 51kB ...
    u10         (valid_time, latitude, longitude) float32 51kB ...
    v10         (valid_time, latitude, longitude) float32 51kB ...
    ssr         (valid_time, latitude, longitude) float32 51kB ...
    str         (valid_time, latitude, longitude) float32 

C:\Users\mpdox\AppData\Local\Temp\ipykernel_28508\217017058.py:9: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("Dimensions:", dict(ds.dims))


In [7]:
import numpy as np
import xarray as xr
import pandas as pd
import os

# ── ค่าคงที่ ──────────────────────────────────────────────
ELEV_M   = 400.0   # ความสูงเฉลี่ยพื้นที่ศึกษา (เมตร) — ปรับตามจริง
ALPHA    = 0.23    # albedo (FAO-56 reference grass)
MJ_DAY   = 1e-6    # J → MJ

def kelvin_to_celsius(k):
    return k - 273.15

def saturation_vp(T_c):
    """es (kPa) จาก temperature (°C)"""
    return 0.6108 * np.exp(17.27 * T_c / (T_c + 237.3))

def slope_vp(T_c):
    """Δ (kPa/°C)"""
    return 4098 * saturation_vp(T_c) / (T_c + 237.3)**2

def psychrometric_const(elev_m):
    """γ (kPa/°C) จาก elevation"""
    P = 101.3 * ((293 - 0.0065 * elev_m) / 293) ** 5.26  # kPa
    return 0.000665 * P

def wind_2m(u10, v10):
    """แปลง wind 10m → 2m (FAO-56 eq. 47)"""
    ws10 = np.sqrt(u10**2 + v10**2)
    return ws10 * (4.87 / np.log(67.8 * 10 - 5.42))

def compute_eto_daily(ds, elev_m=ELEV_M):
    """
    คำนวณ ETo (mm/day) จาก ERA5-Land dataset
    Input : xr.Dataset with valid_time, latitude, longitude
    Output: xr.DataArray ETo (mm/day)
    """
    # ── อุณหภูมิ ──────────────────────────────────────────
    T_c  = kelvin_to_celsius(ds['t2m'])          # °C
    Td_c = kelvin_to_celsius(ds['d2m'])          # °C dewpoint

    # ── Vapour pressure ───────────────────────────────────
    es = saturation_vp(T_c)                      # kPa
    ea = saturation_vp(Td_c)                     # kPa (actual)
    delta = slope_vp(T_c)                        # kPa/°C
    gamma = psychrometric_const(elev_m)          # kPa/°C (scalar)

    # ── Radiation → MJ/m²/day ─────────────────────────────
    Rs  =  ds['ssr'] * MJ_DAY                    # shortwave ↓
    Rnl = -ds['str'] * MJ_DAY                    # net longwave (ERA5 str เป็น negative → กลับเครื่องหมาย)
    Rns = (1 - ALPHA) * Rs                       # net shortwave
    Rn  = Rns - Rnl                              # net radiation
    G   = xr.zeros_like(Rn)                      # soil heat flux ≈ 0 (daily)

    # ── Wind 10m → 2m ─────────────────────────────────────
    u2 = wind_2m(ds['u10'], ds['v10'])

    # ── FAO-56 Penman-Monteith ────────────────────────────
    numerator   = (0.408 * delta * (Rn - G)
                   + gamma * (900 / (T_c + 273)) * u2 * (es - ea))
    denominator = delta + gamma * (1 + 0.34 * u2)
    ETo = numerator / denominator

    ETo = ETo.rename('ETo')
    ETo.attrs.update({'units': 'mm/day', 'long_name': 'Reference Evapotranspiration (FAO-56 PM)'})
    return ETo

# ── ทดสอบกับ 2020 ─────────────────────────────────────────
DST_DIR = r"c:\Users\mpdox\Desktop\ETo"   # ← แก้ path

ds2020 = xr.open_dataset(os.path.join(DST_DIR, 'era5_2020.nc'), engine='netcdf4')

ETo_2020 = compute_eto_daily(ds2020)
print(ETo_2020)
print("\nStats (spatial mean per day):")
print(ETo_2020.mean(['latitude','longitude']).to_series().describe().round(2))

<xarray.DataArray 'ETo' (valid_time: 366, latitude: 5, longitude: 7)> Size: 102kB
array([[[1.96457455, 2.02816747, 2.40502921, ..., 2.67343777,
         2.75935596, 2.8809789 ],
        [1.9541912 , 1.98855219, 2.35283937, ..., 2.60253626,
         2.62839051, 2.83687594],
        [2.17914439, 2.06949728, 2.33149322, ..., 2.48364293,
         2.54556976, 2.69698193],
        [2.27958421, 2.14917828, 2.22251655, ..., 2.47158044,
         2.5121456 , 2.67707562],
        [2.29300874, 2.25387101, 2.31512308, ..., 2.44782318,
         2.49431135, 2.73121684]],

       [[1.92350941, 1.97009019, 2.20978027, ..., 2.41650058,
         2.51546059, 2.60978732],
        [1.9373192 , 1.96127076, 2.22694676, ..., 2.42925301,
         2.48125278, 2.69457534],
        [2.18906179, 2.10386855, 2.38469606, ..., 2.47581098,
         2.53614877, 2.68338139],
        [2.36367547, 2.31317256, 2.38785121, ..., 2.64483992,
         2.64847404, 2.79320993],
        [2.43588863, 2.5006924 , 2.57058867, ..., 2.

In [8]:
import glob

YEARS = range(2020, 2024)

eto_list = []

for yr in YEARS:
    fpath = os.path.join(DST_DIR, f'era5_{yr}.nc')
    if not os.path.exists(fpath):
        print(f"⚠️  ไม่พบไฟล์: {fpath} — ข้าม")
        continue

    ds = xr.open_dataset(fpath, engine='netcdf4')
    eto = compute_eto_daily(ds)
    eto_list.append(eto)
    print(f"✅ {yr}  |  mean={float(eto.mean()):.2f}  min={float(eto.min()):.2f}  max={float(eto.max()):.2f}  mm/day")

# ── Concatenate ───────────────────────────────────────────
ETo_all = xr.concat(eto_list, dim='valid_time')
ETo_all = ETo_all.sortby('valid_time')

print(f"\nรวม : {len(ETo_all.valid_time)} days  "
      f"({str(ETo_all.valid_time.values[0])[:10]} → "
      f"{str(ETo_all.valid_time.values[-1])[:10]})")

# ── บันทึก ────────────────────────────────────────────────
out_path = os.path.join(DST_DIR, 'ETo_2020_2024.nc')
ETo_all.to_netcdf(out_path)
print(f"💾 บันทึกแล้ว: {out_path}")

✅ 2020  |  mean=3.50  min=0.45  max=7.64  mm/day
✅ 2021  |  mean=3.17  min=0.41  max=7.21  mm/day
✅ 2022  |  mean=3.10  min=0.53  max=6.08  mm/day
✅ 2023  |  mean=3.44  min=0.48  max=7.23  mm/day

รวม : 1461 days  (2020-01-01 → 2023-12-31)
💾 บันทึกแล้ว: c:\Users\mpdox\Desktop\ETo\ETo_2020_2024.nc


In [10]:
import xarray as xr
import numpy as np
import pandas as pd
import os

# ════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════
DST_DIR  = r"c:\Users\mpdox\Desktop\ETo"   # ← แก้ path
YEARS    = range(2018, 2025)
ELEV_M   = 300.0
ALPHA    = 0.23
MJ_DAY   = 1e-6

# ════════════════════════════════════════════════════════════
# FUNCTIONS (เหมือนเดิม)
# ════════════════════════════════════════════════════════════
def kelvin_to_celsius(k):   return k - 273.15
def saturation_vp(T_c):     return 0.6108 * np.exp(17.27 * T_c / (T_c + 237.3))
def slope_vp(T_c):          return 4098 * saturation_vp(T_c) / (T_c + 237.3)**2
def psychrometric_const(e): P = 101.3*((293-0.0065*e)/293)**5.26; return 0.000665*P
def wind_2m(u10, v10):      return np.sqrt(u10**2+v10**2) * (4.87/np.log(67.8*10-5.42))

def compute_eto_daily(ds, elev_m=ELEV_M):
    T_c   = kelvin_to_celsius(ds['t2m'])
    Td_c  = kelvin_to_celsius(ds['d2m'])
    es    = saturation_vp(T_c)
    ea    = saturation_vp(Td_c)
    delta = slope_vp(T_c)
    gamma = psychrometric_const(elev_m)
    Rs    =  ds['ssr'] * MJ_DAY
    Rnl   = -ds['str'] * MJ_DAY
    Rn    = (1 - ALPHA) * Rs - Rnl
    u2    = wind_2m(ds['u10'], ds['v10'])
    ETo   = (0.408*delta*(Rn) + gamma*(900/(T_c+273))*u2*(es-ea)) / \
            (delta + gamma*(1+0.34*u2))
    return ETo, T_c, ea, es, u2, Rn

# ════════════════════════════════════════════════════════════
# STEP 1 — คำนวณ ETo ทุกปี → spatial mean → daily DataFrame
# ════════════════════════════════════════════════════════════
records = []

for yr in YEARS:
    fpath = os.path.join(DST_DIR, f'era5_{yr}.nc')
    if not os.path.exists(fpath):
        print(f"⚠️  ไม่พบ: {fpath}")
        continue

    ds = xr.open_dataset(fpath, engine='netcdf4')

    ETo, T_c, ea, es, u2, Rn = compute_eto_daily(ds)

    # spatial mean ทั้ง grid (5×7 points)
    sp = dict(latitude=ds.latitude, longitude=ds.longitude)
    def smean(da): return float(da.mean(['latitude','longitude']))

    for i, t in enumerate(ds['valid_time'].values):
        date = pd.Timestamp(t)
        T_i  = float(T_c.isel(valid_time=i).mean(['latitude','longitude']))
        ea_i = float(ea.isel(valid_time=i).mean(['latitude','longitude']))
        es_i = float(es.isel(valid_time=i).mean(['latitude','longitude']))
        u2_i = float(u2.isel(valid_time=i).mean(['latitude','longitude']))
        Rn_i = float(Rn.isel(valid_time=i).mean(['latitude','longitude']))
        et_i = float(ETo.isel(valid_time=i).mean(['latitude','longitude']))
        RH_i = min(100.0, (ea_i / es_i) * 100)
        VPD_i = max(0.0, es_i - ea_i)

        records.append({
            'date'      : date.date(),
            'year'      : date.year,
            'doy'       : date.day_of_year,
            'T_mean'    : round(T_i,   2),
            'RH_pct'    : round(RH_i,  1),
            'VPD_kPa'   : round(VPD_i, 4),
            'u2_ms'     : round(u2_i,  3),
            'Rn_MJ'     : round(Rn_i,  4),
            'ETo_mm_day': round(et_i,  3),
        })

    ds.close()
    print(f"✅ {yr}  ETo mean={np.mean([r['ETo_mm_day'] for r in records if r['year']==yr]):.2f} mm/day")

df_daily = pd.DataFrame(records)
df_daily['date'] = pd.to_datetime(df_daily['date'])

# ════════════════════════════════════════════════════════════
# STEP 2 — Aggregate รายสัปดาห์ (ISO week)
# ════════════════════════════════════════════════════════════
df_daily['week']     = df_daily['date'].dt.isocalendar().week.astype(int)
df_daily['iso_year'] = df_daily['date'].dt.isocalendar().year.astype(int)

# ใช้ iso_year แทน year เพื่อป้องกัน week 52/53 ข้ามปี
ET0_weekly = (
    df_daily
    .groupby(['iso_year', 'week'])
    .agg(
        ET0_mm_week = ('ETo_mm_day', 'sum'),
        T_mean      = ('T_mean',     'mean'),
        RH_pct      = ('RH_pct',     'mean'),
        VPD_kPa     = ('VPD_kPa',    'mean'),
        u2_ms       = ('u2_ms',      'mean'),
        Rn_MJ       = ('Rn_MJ',      'mean'),
        n_days      = ('ETo_mm_day', 'count'),
    )
    .reset_index()
    .rename(columns={'iso_year': 'year'})
    .round({'ET0_mm_week':2, 'T_mean':2, 'RH_pct':1,
            'VPD_kPa':4, 'u2_ms':3, 'Rn_MJ':4})
)

# กรองเฉพาะ week ที่มีข้อมูลครบ 7 วัน (ป้องกัน partial week ต้น/ท้ายปี)
ET0_weekly = ET0_weekly[ET0_weekly['n_days'] == 7].reset_index(drop=True)

# ════════════════════════════════════════════════════════════
# STEP 3 — Sanity Check
# ════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("SANITY CHECK — ETo Weekly")
print("═"*55)
print(f"ช่วงเวลา   : {ET0_weekly['year'].min()} W{ET0_weekly['week'].min():02d}"
      f" → {ET0_weekly['year'].max()} W{ET0_weekly['week'].max():02d}")
print(f"จำนวน weeks: {len(ET0_weekly)}")
print(f"\n{'Metric':<15} {'mm/week':>10} {'หมายเหตุ'}")
print("-"*55)
print(f"{'Mean':<15} {ET0_weekly['ET0_mm_week'].mean():>10.2f}  ควรอยู่ ~20–28 mm/week")
print(f"{'Std':<15} {ET0_weekly['ET0_mm_week'].std():>10.2f}")
print(f"{'Min':<15} {ET0_weekly['ET0_mm_week'].min():>10.2f}  wet season (ฝนตก)")
print(f"{'Max':<15} {ET0_weekly['ET0_mm_week'].max():>10.2f}  ควร < 50 mm/week")
print(f"{'Missing weeks':<15} {(ET0_weekly['n_days']<7).sum():>10}  ควร = 0")

# ตรวจ negative / outlier
neg = (ET0_weekly['ET0_mm_week'] < 0).sum()
ext = (ET0_weekly['ET0_mm_week'] > 55).sum()
print(f"{'Negative':<15} {neg:>10}  ควร = 0")
print(f"{'Extreme >55':<15} {ext:>10}  ควร = 0")

# Seasonal pattern check
ET0_weekly['season'] = ET0_weekly['week'].apply(
    lambda w: 'Dry(hot)' if 10<=w<=18 else ('Wet' if 22<=w<=40 else 'Dry(cool)'))
print("\nค่าเฉลี่ยตามฤดูกาล:")
print(ET0_weekly.groupby('season')['ET0_mm_week']
      .agg(['mean','min','max']).round(2).to_string())

# ════════════════════════════════════════════════════════════
# STEP 4 — ตรวจ gap (สัปดาห์ขาดหาย)
# ════════════════════════════════════════════════════════════
all_weeks = set()
for yr in ET0_weekly['year'].unique():
    for wk in range(1, 53):
        all_weeks.add((yr, wk))
existing = set(zip(ET0_weekly['year'], ET0_weekly['week']))
missing  = sorted(all_weeks - existing)
if missing:
    print(f"\n⚠️  Weeks ที่ขาดหาย ({len(missing)} weeks):")
    for y, w in missing[:10]: print(f"   {y} W{w:02d}")
else:
    print("\n✅ ไม่มี week ขาดหาย")

# ════════════════════════════════════════════════════════════
# STEP 5 — Export
# ════════════════════════════════════════════════════════════
df_daily.to_csv('ET0_daily_phayao_2018_2024.csv', index=False)
ET0_weekly[['year','week','ET0_mm_week','T_mean','RH_pct',
            'VPD_kPa','u2_ms','Rn_MJ','n_days']].to_csv(
    'ET0_weekly_phayao_2018_2024.csv', index=False)

print("\n✅ Saved: ET0_daily_phayao_2018_2024.csv")
print("✅ Saved: ET0_weekly_phayao_2018_2024.csv")
print(f"\nตัวอย่างข้อมูล (week 26–30 ปี 2022 = wet season peak):")
sample = ET0_weekly[(ET0_weekly['year']==2022) &
                    (ET0_weekly['week'].between(26,30))]
print(sample[['year','week','ET0_mm_week','T_mean',
              'RH_pct','VPD_kPa']].to_string(index=False))

print("\n📌 NEXT: รัน phase3_step2_chirps_rainfall.py")

⚠️  ไม่พบ: c:\Users\mpdox\Desktop\ETo\era5_2018.nc
⚠️  ไม่พบ: c:\Users\mpdox\Desktop\ETo\era5_2019.nc
✅ 2020  ETo mean=3.50 mm/day
✅ 2021  ETo mean=3.17 mm/day
✅ 2022  ETo mean=3.10 mm/day
✅ 2023  ETo mean=3.44 mm/day
✅ 2024  ETo mean=3.39 mm/day

═══════════════════════════════════════════════════════
SANITY CHECK — ETo Weekly
═══════════════════════════════════════════════════════
ช่วงเวลา   : 2020 W01 → 2024 W53
จำนวน weeks: 260

Metric             mm/week หมายเหตุ
-------------------------------------------------------
Mean                 23.25  ควรอยู่ ~20–28 mm/week
Std                   7.40
Min                   9.66  wet season (ฝนตก)
Max                  46.02  ควร < 50 mm/week
Missing weeks            0  ควร = 0
Negative                 0  ควร = 0
Extreme >55              0  ควร = 0

ค่าเฉลี่ยตามฤดูกาล:
            mean    min    max
season                        
Dry(cool)  21.40  11.26  34.88
Dry(hot)   33.94  20.30  46.02
Wet        20.52   9.66  35.29

⚠️  Weeks ที่ขาดห